In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================================
# CODET5 + PRIMEVUL : FINAL ONE-CELL KAGGLE CODE
# ============================================================

# --------------------
# Imports
# --------------------
import os
import json
import torch
import numpy as np
import pandas as pd
import platform
import psutil
from datetime import datetime

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------
# DEVICE
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# HARDWARE INFO
# ============================================================

hardware_info = {
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "None",
    "torch_version": torch.__version__,
    "ram_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2),
    "os": platform.system(),
    "platform": platform.platform()
}

print("\nHardware Info:")
for k, v in hardware_info.items():
    print(f"{k}: {v}")

# ============================================================
# DATASET PATH (✅ FIXED)
# ============================================================

DATASET_PATH = "/kaggle/input/dataset-primevult5"

print("\nFiles in dataset:")
files = os.listdir(DATASET_PATH)
for f in files:
    print(" -", f)

# ============================================================
# LOAD PRIMEVUL JSONL
# ============================================================

def load_primevul_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line.strip())
            data.append({
                "code": obj["func"],
                "label": int(obj["target"])
            })
    return pd.DataFrame(data)

print("\nLoading PrimeVul dataset...")

train_df = load_primevul_jsonl(f"{DATASET_PATH}/primevul_train_paired.jsonl")
val_df   = load_primevul_jsonl(f"{DATASET_PATH}/primevul_valid_paired.jsonl")
test_df  = load_primevul_jsonl(f"{DATASET_PATH}/primevul_test_paired.jsonl")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("\nTrain label distribution:\n", train_df["label"].value_counts())

# ============================================================
# HF DATASETS
# ============================================================

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df, preserve_index=False)
test_ds  = Dataset.from_pandas(test_df, preserve_index=False)

# ============================================================
# TOKENIZER (CORRECT FOR CODET5)
# ============================================================

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")

def tokenize_fn(batch):
    return tokenizer(
        batch["code"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch", columns=cols)
test_ds.set_format("torch", columns=cols)

# ============================================================
# MODEL
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    "Salesforce/codet5-base",
    num_labels=2
).to(device)

# ============================================================
# TRAINING ARGUMENTS (LOW LOSS)
# ============================================================

training_args = TrainingArguments(
    output_dir="./codet5_primevul",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    gradient_accumulation_steps=2,   # effective batch = 16
    save_strategy="no",
    logging_steps=100,
    report_to="none"
)

# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

# ============================================================
# TRAIN
# ============================================================

print("\nStarting CodeT5 training on PrimeVul...")
trainer.train()

# ============================================================
# FINAL EVALUATION
# ============================================================

print("\nEvaluating on test set...")

preds = trainer.predict(test_ds)

logits = preds.predictions
if isinstance(logits, tuple):
    logits = logits[0]

y_true = preds.label_ids
y_pred = np.argmax(logits, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

print("\n===== FINAL CODET5 PRIMEVUL RESULTS =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("FPR      :", fpr)
print("Confusion Matrix:", tn, fp, fn, tp)

# ============================================================
# SAVE RESULTS + HARDWARE
# ============================================================

results = {
    "dataset": "PrimeVul",
    "model": "CodeT5",
    "epochs": 6,
    "learning_rate": 1e-5,
    "effective_batch_size": 16,
    "metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr)
    },
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    },
    "hardware": hardware_info,
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

out_path = "/kaggle/working/CodeT5_PrimeVul_results.json"
with open(out_path, "w") as f:
    json.dump(results, f)

print("\nResults saved to:", out_path)


Device: cuda
GPU: Tesla T4

Hardware Info:
gpu: Tesla T4
cuda_version: 12.6
torch_version: 2.8.0+cu126
ram_gb: 31.35
os: Linux
platform: Linux-6.6.113+-x86_64-with-glibc2.35

Files in dataset:
 - primevul_test_paired.jsonl
 - primevul_train_paired.jsonl
 - primevul_valid_paired.jsonl

Loading PrimeVul dataset...
Train: (7578, 2)
Val  : (960, 2)
Test : (870, 2)

Train label distribution:
 label
1    3789
0    3789
Name: count, dtype: int64


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...


Starting CodeT5 training on PrimeVul...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,0.815200
200,0.795400
300,0.774600
400,0.742700
500,0.723700
600,0.728800
700,0.715800
800,0.726900
900,0.721500
1000,0.715900



Evaluating on test set...



===== FINAL CODET5 PRIMEVUL RESULTS =====
Accuracy : 0.5149425287356322
Precision: 0.5150115473441108
Recall   : 0.5126436781609195
F1 Score : 0.5138248847926268
FPR      : 0.4827586206896552
Confusion Matrix: 225 210 212 223

Results saved to: /kaggle/working/CodeT5_PrimeVul_results.json
